# Frequent Itemset Mining

Learning Objectives:

- Extract frequent patterns given a corpus of data.
- Find the rules which are interesting and non-obvious for a given domain.

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

df = pd.read_excel('Online Retail.xlsx')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
df2 = pd.read_excel('online_retail_II.xlsx')
df2.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Question 1.1: Concatinate both dataframes to create a single dataframe. Remove any rows where InvoiceNo is Null and Quantity is Negative

In [3]:
frames = df,df2
data = pd.concat(frames)

In [4]:
data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Invoice,Price,Customer ID
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,NaN,NaN,NaN
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,NaN,NaN,NaN
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,NaN,NaN,NaN
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,NaN,NaN,NaN
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
525456,NaN,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00,NaN,NaN,United Kingdom,538171,2.95,17530.0
525457,NaN,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00,NaN,NaN,United Kingdom,538171,3.75,17530.0
525458,NaN,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00,NaN,NaN,United Kingdom,538171,3.75,17530.0
525459,NaN,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00,NaN,NaN,United Kingdom,538171,3.75,17530.0


In [5]:
data.isna().sum()

InvoiceNo      525461
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
UnitPrice      525461
CustomerID     660541
Country             0
Invoice        541909
Price          541909
Customer ID    649836
dtype: int64

In [6]:
data = data[data["Quantity"]>0.0]
data = data.dropna(axis="index",subset="InvoiceNo")
data['InvoiceNo'] = data['InvoiceNo'].astype(str)
data['Description'] = data['Description'].str.strip()
data["StockCode"] = data["StockCode"].astype(str)

In [7]:
data.isna().sum()

InvoiceNo           0
StockCode           0
Description       592
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     133361
Country             0
Invoice        531285
Price          531285
Customer ID    531285
dtype: int64

## Question 1.2: Filter the data by only transactions that happened in United Kingdom 

In [8]:
data["Country"].value_counts()

United Kingdom          486286
Germany                   9042
France                    8408
EIRE                      7894
Spain                     2485
Netherlands               2363
Belgium                   2031
Switzerland               1967
Portugal                  1501
Australia                 1185
Norway                    1072
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Unspecified                446
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     295
Hong Kong                  284
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         58
Lebanon 

In [9]:
data = data[data["Country"]=="United Kingdom"]

In [10]:
data["Country"].value_counts()

United Kingdom    486286
Name: Country, dtype: int64

## Question 1.3: What are the most popular 5 items?

In [11]:
data["Description"].value_counts()[:5]

WHITE HANGING HEART T-LIGHT HOLDER    2231
JUMBO BAG RED RETROSPOT               1960
REGENCY CAKESTAND 3 TIER              1711
PARTY BUNTING                         1615
LUNCH BAG RED RETROSPOT               1421
Name: Description, dtype: int64

In [12]:
data["StockCode"].value_counts()[:5]

85123A    2174
85099B    1960
22423     1711
47566     1615
20725     1421
Name: StockCode, dtype: int64

## Question 1.4: Filter down the data to include transaction that contain the top 20 items

In [13]:
top20 = data["Description"].value_counts()[:20].index.astype(str)

In [14]:
data_top20 = []
for item in top20:
    data_top20.append(data[data["Description"]==item])
data_top20 = pd.concat(data_top20, ignore_index=True)

In [15]:
data_top20

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Invoice,Price,Customer ID
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,NaN,NaN,NaN
1,536373,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:02:00,2.55,17850.0,United Kingdom,NaN,NaN,NaN
2,536375,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 09:32:00,2.55,17850.0,United Kingdom,NaN,NaN,NaN
3,536390,85123A,WHITE HANGING HEART T-LIGHT HOLDER,64,2010-12-01 10:19:00,2.55,17511.0,United Kingdom,NaN,NaN,NaN
4,536394,85123A,WHITE HANGING HEART T-LIGHT HOLDER,32,2010-12-01 10:39:00,2.55,13408.0,United Kingdom,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
26445,581219,22383,LUNCH BAG SUKI DESIGN,2,2011-12-08 09:28:00,4.13,NaN,United Kingdom,NaN,NaN,NaN
26446,581256,22383,LUNCH BAG SUKI DESIGN,12,2011-12-08 11:21:00,4.96,NaN,United Kingdom,NaN,NaN,NaN
26447,581334,22383,LUNCH BAG SUKI DESIGN,5,2011-12-08 12:07:00,1.65,17841.0,United Kingdom,NaN,NaN,NaN
26448,581453,22383,LUNCH BAG SUKI DESIGN,10,2011-12-08 18:15:00,1.65,16401.0,United Kingdom,NaN,NaN,NaN


## Question 2.1: Consolidate the items into 1 transaction per row and each product one-hot encoded.

In [16]:
transactions = data_top20["InvoiceNo"].value_counts().index #array of invoice numbers for every transaction that includes at least one of the top 20 items

In [17]:
InvoiceNo = transactions
StockCode = [[data_top20[data_top20["InvoiceNo"]==t]["StockCode"].value_counts().index.values.tolist() ][0] for t in transactions] #array of stockids for the items purchased 
Description = [[data_top20[data_top20["InvoiceNo"]==t]["Description"].value_counts().index.values.tolist() ][0] for t in transactions]
Quantity  = [[data_top20[data_top20["InvoiceNo"]==t]["StockCode"].value_counts().values] for t in transactions]

In [18]:
dfs = [pd.DataFrame(columns=["InvoiceNo"]+top20.tolist())]
for transaction in transactions:
    i = transactions.tolist().index(transaction)
    df = pd.DataFrame(Quantity[i],columns=Description[i])
    df.insert(0,"InvoiceNo",transaction)
    dfs.append(df)
df = pd.concat(dfs,axis=0,ignore_index=True)
df = df.fillna(value=0)

In [19]:
#Create the "basket"
basket = df

## Question 2.2: Convert all the values to 1 when values are greater than 0 and 0 when values are 0 or less.

In [20]:
for feature in basket.columns[1:]:
    basket[feature] = basket[feature] > 0
basket

,InvoiceNo,WHITE HANGING HEART T-LIGHT HOLDER,JUMBO BAG RED RETROSPOT,REGENCY CAKESTAND 3 TIER,PARTY BUNTING,LUNCH BAG RED RETROSPOT,ASSORTED COLOUR BIRD ORNAMENT,LUNCH BAG BLACK SKULL.,SET OF 3 CAKE TINS PANTRY DESIGN,NATURAL SLATE HEART CHALKBOARD,...,JUMBO BAG PINK POLKADOT,PACK OF 72 RETROSPOT CAKE CASES,PAPER CHAIN KIT 50'S CHRISTMAS,JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO STORAGE BAG SUKI,LUNCH BAG CARS BLUE,WOODEN PICTURE FRAME WHITE FINISH,LUNCH BAG SPACEBOY DESIGN,SPOTTY BUNTING,LUNCH BAG SUKI DESIGN
0,573585,True,True,True,True,True,False,True,True,True,...,True,True,True,True,True,True,True,True,True,True
1,577504,True,True,False,False,True,False,True,True,False,...,True,True,True,False,True,True,False,True,False,True
2,580729,True,True,True,False,True,True,True,True,True,...,False,True,True,False,True,True,True,True,True,True
3,536876,True,True,True,True,True,True,True,False,True,...,False,True,True,False,True,False,True,True,False,False
4,564838,True,True,True,True,True,False,True,True,True,...,True,False,True,True,True,True,True,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9961,555592,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9962,555577,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9963,554943,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9964,554347,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## Question 3.1: Apply [apriori](http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/) algorithm to generate frequent item sets that have a support of at least 7%

In [21]:
from mlxtend.frequent_patterns import apriori, association_rules

In [22]:
basket_cut = basket.drop(columns="InvoiceNo")

In [23]:
frq_items = apriori(basket_cut, min_support=0.07, use_colnames=True)
frq_items

,support,itemsets
0,0.217339,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.194461,(JUMBO BAG RED RETROSPOT)
2,0.169075,(REGENCY CAKESTAND 3 TIER)
3,0.159944,(PARTY BUNTING)
4,0.139675,(LUNCH BAG RED RETROSPOT)
5,0.137568,(ASSORTED COLOUR BIRD ORNAMENT)
6,0.122015,(LUNCH BAG BLACK SKULL.)
7,0.124523,(SET OF 3 CAKE TINS PANTRY DESIGN)
8,0.122316,(NATURAL SLATE HEART CHALKBOARD)
9,0.116797,(HEART OF WICKER SMALL)


## Question 3.2: Generate the association rules with their corresponding support, confidence and lift.

In [24]:
association_rules(frq_items, metric="support", min_threshold=0.1)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric


In [25]:
association_rules(frq_items, metric="confidence", min_threshold=0.1)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(JUMBO BAG PINK POLKADOT),(JUMBO BAG RED RETROSPOT),0.116295,0.194461,0.078768,0.677308,3.482999,0.056153,2.496309,0.806707
1,(JUMBO BAG RED RETROSPOT),(JUMBO BAG PINK POLKADOT),0.194461,0.116295,0.078768,0.405057,3.482999,0.056153,1.485360,0.884987
2,(JUMBO BAG RED RETROSPOT),(JUMBO STORAGE BAG SUKI),0.194461,0.113386,0.070038,0.360165,3.176465,0.047989,1.385693,0.850592
3,(JUMBO STORAGE BAG SUKI),(JUMBO BAG RED RETROSPOT),0.113386,0.194461,0.070038,0.617699,3.176465,0.047989,2.107081,0.772810


In [26]:
association_rules(frq_items, metric="lift", min_threshold=0.1)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(JUMBO BAG PINK POLKADOT),(JUMBO BAG RED RETROSPOT),0.116295,0.194461,0.078768,0.677308,3.482999,0.056153,2.496309,0.806707
1,(JUMBO BAG RED RETROSPOT),(JUMBO BAG PINK POLKADOT),0.194461,0.116295,0.078768,0.405057,3.482999,0.056153,1.485360,0.884987
2,(JUMBO BAG RED RETROSPOT),(JUMBO STORAGE BAG SUKI),0.194461,0.113386,0.070038,0.360165,3.176465,0.047989,1.385693,0.850592
3,(JUMBO STORAGE BAG SUKI),(JUMBO BAG RED RETROSPOT),0.113386,0.194461,0.070038,0.617699,3.176465,0.047989,2.107081,0.772810


In [27]:
rules = association_rules(frq_items, metric ="lift", min_threshold = 0.1)
rules = rules.sort_values(['confidence', 'lift'], ascending =[False, False])
rules.sort_index()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction,zhangs_metric
0,(JUMBO BAG PINK POLKADOT),(JUMBO BAG RED RETROSPOT),0.116295,0.194461,0.078768,0.677308,3.482999,0.056153,2.496309,0.806707
1,(JUMBO BAG RED RETROSPOT),(JUMBO BAG PINK POLKADOT),0.194461,0.116295,0.078768,0.405057,3.482999,0.056153,1.485360,0.884987
2,(JUMBO BAG RED RETROSPOT),(JUMBO STORAGE BAG SUKI),0.194461,0.113386,0.070038,0.360165,3.176465,0.047989,1.385693,0.850592
3,(JUMBO STORAGE BAG SUKI),(JUMBO BAG RED RETROSPOT),0.113386,0.194461,0.070038,0.617699,3.176465,0.047989,2.107081,0.772810


## Question 4: Based on the above rules, identify what would be the opportunity of promoting one of the antecendents.

#### It seems that it's more likely for customers to purchase more than one "Jumbo Bag", but more specifically, it seems the "Red Retrospot" color and the "Pink Polkadot" color are often purchased together. In addition, the "Jumbo Storage Bag Suki" is also often purchased along side the two popular color variants of the "Jumbo Bag" items. I'm not sure what the "Suki" suffix suggests but pherhaps it's a color variant of the "Storage Bag" items.

## Question 5. Create a new text cell in your Notebook: Complete a 50-100 word summary (or short description of your thinking in applying this week's learning to the solution) of your experience in this assignment. Include:
                                                                      
* What was your incoming experience with this model, if any? 
* What steps you took, what obstacles you encountered.
* How you link this exercise to real-world, machine learning problem-solving. (What steps were missing? What else do you need to learn?) 
> This summary allows your instructor to know how you are doing and allot points for your effort in thinking and planning, and making connections to real-world work.


#### Like previous weeks, I didn't have any incoming experience with "apriori" or "association_rules". Generally it was all new to me besides what I've learned so far, so naturally I did run into some issues when intially using them but beyond that I had relatively no issues completing this assignment. I made full use of the API Documentation when working through the assignment since it was all new, thankfully it was sufficient for my purposes. I could see how this process of exploring patterns in frequent itemsets to obtian insight on the connections within the dataset would be done in a number of fields where ML is applied, if the amount of time and effort spent searching for a better approach is less expensive relative to the monetary increases that would result from such an improvement than this is more than a fruitful effort. I apologize for being repetitive here by being unable to give a real-world example, however, again this is just due to my lack of experience in the field and in applying ML. I really do hope to apply these skills to a career after this course is finished.